<a href="https://colab.research.google.com/github/LizaHam123/sentiment-analysis-algerie-telecom/blob/main/Nettoyage_commentaires.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Installation de spaCy et du modèle français
!pip install spacy pandas tqdm
!python -m spacy download fr_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 91.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import pandas as pd
import spacy
import re
import string
from tqdm import tqdm
import time

In [ ]:
# Activer la barre de progression pour pandas
tqdm.pandas()

In [ ]:
print("Chargement du modèle spaCy français...")
nlp = spacy.load("fr_core_news_sm")
print("Modèle chargé avec succès!")

Chargement du modèle spaCy français...
Modèle chargé avec succès!


In [ ]:
from google.colab import files
import io

# Upload du fichier
print("Veuillez sélectionner votre fichier CSV:")
uploaded = files.upload()

# Récupérer le nom du fichier uploadé
filename = list(uploaded.keys())[0]
print(f"Fichier '{filename}' uploadé avec succès!")

# Charger le CSV
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f"Données chargées: {len(df)} lignes, {len(df.columns)} colonnes")
print("\nAperçu des données:")
print(df.head())

Veuillez sélectionner votre fichier CSV:


Saving resultats_analysef.csv to resultats_analysef.csv
Fichier 'resultats_analysef.csv' uploadé avec succès!
Données chargées: 9238 lignes, 3 colonnes

Aperçu des données:
                                         commentaire sentiment  \
0                       Plantez et vous récolterez.,   positif   
1  Bravo et merci à tous les employés d’Algérie T...   positif   
2  Votre entreprise ne pense plus qu’aux abonnés ...   positif   
3  Ce qui nous réjouit, c’est le passage de Idoom...   positif   
4  On demande juste à être raccordés à la nouvell...   négatif   

          probleme  
0              NaN  
1              NaN  
2              NaN  
3              NaN  
4  problème réseau  


In [ ]:
# ============================================================================
# Fonctions de nettoyage
# ============================================================================

In [ ]:
def nettoyer_texte_basique(texte):
    """
    Nettoyage basique du texte (rapide)
    """
    if pd.isna(texte) or texte == "":
        return ""

    # Convertir en string si ce n'est pas déjà le cas
    texte = str(texte)

    # Supprimer les guillemets en début et fin
    texte = texte.strip('"\'')

    # Supprimer les virgules en fin de phrase
    texte = re.sub(r'[,;]+$', '', texte)

    # Normaliser les espaces
    texte = re.sub(r'\s+', ' ', texte)

    # Supprimer les espaces en début et fin
    texte = texte.strip()

    return texte

In [ ]:
def nettoyer_avec_spacy(texte):
    """
    Nettoyage avancé avec spaCy (plus lent mais plus précis)
    """
    if pd.isna(texte) or texte == "":
        return {"texte_nettoye": "", "lemmes": "", "tokens_utiles": ""}

    # Nettoyage basique d'abord
    texte = nettoyer_texte_basique(texte)

    if texte == "":
        return {"texte_nettoye": "", "lemmes": "", "tokens_utiles": ""}

    # Traitement avec spaCy
    doc = nlp(texte)

    # Extraire les lemmes (forme canonique des mots)
    lemmes = []
    tokens_utiles = []

    for token in doc:
        # Ignorer la ponctuation, les espaces et les mots vides
        if not token.is_punct and not token.is_space and not token.is_stop:
            lemmes.append(token.lemma_.lower())
            tokens_utiles.append(token.text.lower())

    return {
        "texte_nettoye": texte,
        "lemmes": " ".join(lemmes),
        "tokens_utiles": " ".join(tokens_utiles)
    }



In [ ]:
print("Début du nettoyage basique...")
start_time = time.time()

# Appliquer le nettoyage basique avec barre de progression (SANS desc=)
df['commentaire_nettoye'] = df['commentaire'].progress_apply(
    nettoyer_texte_basique
)

end_time = time.time()
print(f"Nettoyage basique terminé en {end_time - start_time:.2f} secondes")

# Afficher quelques exemples
print("\n📋 Exemples de nettoyage basique:")
for i in range(min(5, len(df))):
    print(f"Original: {df.iloc[i]['commentaire']}")
    print(f"Nettoyé:  {df.iloc[i]['commentaire_nettoye']}")
    print("-" * 50)

Début du nettoyage basique...


100%|██████████| 9238/9238 [00:00<00:00, 42184.37it/s]

Nettoyage basique terminé en 0.22 secondes

📋 Exemples de nettoyage basique:
Original: Plantez et vous récolterez.,
Nettoyé:  Plantez et vous récolterez.
--------------------------------------------------
Original: Bravo et merci à tous les employés d’Algérie Télécom.,
Nettoyé:  Bravo et merci à tous les employés d’Algérie Télécom.
--------------------------------------------------
Original: Votre entreprise ne pense plus qu’aux abonnés Fibre !!!,
Nettoyé:  Votre entreprise ne pense plus qu’aux abonnés Fibre !!!
--------------------------------------------------
Original: Ce qui nous réjouit, c’est le passage de Idoom ADSL à Idoom Fibre.,
Nettoyé:  Ce qui nous réjouit, c’est le passage de Idoom ADSL à Idoom Fibre.
--------------------------------------------------
Original: On demande juste à être raccordés à la nouvelle technologie de fibre optique, ça fait trop longtemps qu’on attend.,
Nettoyé:  On demande juste à être raccordés à la nouvelle technologie de fibre optique, ça fait tro

In [ ]:
print("Début du nettoyage avancé avec spaCy...")
print("Cette étape peut prendre plusieurs minutes pour 9000+ commentaires...")

start_time = time.time()

# Fonction pour traiter par batch (plus efficace)
def traiter_batch_spacy(textes, batch_size=100):
    """
    Traite les textes par batch pour optimiser spaCy
    """
    resultats = []

    for i in tqdm(range(0, len(textes), batch_size), desc="Traitement spaCy"):
        batch = textes[i:i+batch_size]

        for texte in batch:
            resultats.append(nettoyer_avec_spacy(texte))

    return resultats

# Traitement par batch
resultats_spacy = traiter_batch_spacy(df['commentaire'].tolist())

# Créer les nouvelles colonnes
df['commentaire_spacy'] = [r['texte_nettoye'] for r in resultats_spacy]
df['lemmes'] = [r['lemmes'] for r in resultats_spacy]
df['tokens_utiles'] = [r['tokens_utiles'] for r in resultats_spacy]

end_time = time.time()
print(f"Nettoyage spaCy terminé en {end_time - start_time:.2f} secondes")


Début du nettoyage avancé avec spaCy...
Cette étape peut prendre plusieurs minutes pour 9000+ commentaires...


Traitement spaCy: 100%|██████████| 93/93 [01:23<00:00,  1.11it/s]

Nettoyage spaCy terminé en 83.68 secondes


In [ ]:
print("STATISTIQUES DE NETTOYAGE")
print("=" * 50)

print(f"Nombre total de commentaires: {len(df)}")
print(f"Commentaires vides après nettoyage: {sum(df['commentaire_nettoye'] == '')}")
print(f"Commentaires non vides: {sum(df['commentaire_nettoye'] != '')}")

# Longueur moyenne des commentaires
df['longueur_originale'] = df['commentaire'].astype(str).str.len()
df['longueur_nettoyee'] = df['commentaire_nettoye'].str.len()

print(f"Longueur moyenne originale: {df['longueur_originale'].mean():.1f} caractères")
print(f"Longueur moyenne nettoyée: {df['longueur_nettoyee'].mean():.1f} caractères")

# Distribution par sentiment
print(f"\nDistribution par sentiment:")
print(df['sentiment'].value_counts())

# Exemples de résultats
print(f"\n EXEMPLES DE RÉSULTATS:")
print("=" * 50)

for i in range(min(3, len(df))):
    print(f"Exemple {i+1}:")
    print(f"   Original: {df.iloc[i]['commentaire']}")
    print(f"   Nettoyé:  {df.iloc[i]['commentaire_nettoye']}")
    print(f"   Lemmes:   {df.iloc[i]['lemmes']}")
    print(f"   Tokens:   {df.iloc[i]['tokens_utiles']}")
    print(f"   Sentiment: {df.iloc[i]['sentiment']}")
    print("-" * 50)


STATISTIQUES DE NETTOYAGE
Nombre total de commentaires: 9238
Commentaires vides après nettoyage: 0
Commentaires non vides: 9238
Longueur moyenne originale: 88.9 caractères
Longueur moyenne nettoyée: 87.8 caractères

Distribution par sentiment:
sentiment
négatif    4746
positif    3130
neutre     1362
Name: count, dtype: int64

 EXEMPLES DE RÉSULTATS:
Exemple 1:
   Original: Plantez et vous récolterez.,
   Nettoyé:  Plantez et vous récolterez.
   Lemmes:   plantez récolter
   Tokens:   plantez récolterez
   Sentiment: positif
--------------------------------------------------
Exemple 2:
   Original: Bravo et merci à tous les employés d’Algérie Télécom.,
   Nettoyé:  Bravo et merci à tous les employés d’Algérie Télécom.
   Lemmes:   bravo employé algérie télécom
   Tokens:   bravo employés algérie télécom
   Sentiment: positif
--------------------------------------------------
Exemple 3:
   Original: Votre entreprise ne pense plus qu’aux abonnés Fibre !!!,
   Nettoyé:  Votre entreprise n

In [ ]:
# Créer le DataFrame final avec toutes les colonnes
df_final = df[['commentaire', 'commentaire_nettoye', 'commentaire_spacy',
               'lemmes', 'tokens_utiles', 'sentiment', 'probleme']].copy()

# Sauvegarder en CSV
nom_fichier_sortie = 'commentaires_nettoyes.csv'
df_final.to_csv(nom_fichier_sortie, index=False, encoding='utf-8')

print(f"Fichier sauvegardé: {nom_fichier_sortie}")
print(f"Colonnes dans le fichier final: {list(df_final.columns)}")

# Télécharger le fichier
files.download(nom_fichier_sortie)

print("NETTOYAGE TERMINÉ AVEC SUCCÈS!")
print("Le fichier nettoyé a été téléchargé automatiquement.")

Fichier sauvegardé: commentaires_nettoyes.csv
Colonnes dans le fichier final: ['commentaire', 'commentaire_nettoye', 'commentaire_spacy', 'lemmes', 'tokens_utiles', 'sentiment', 'probleme']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

NETTOYAGE TERMINÉ AVEC SUCCÈS!
Le fichier nettoyé a été téléchargé automatiquement.
